# Multi-Model AI Routing with Nugen

<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="160"/>

## Overview

Different tasks can benefit from different language models. This guide demonstrates **application-level routing**: a Python function classifies a task using simple rules and selects a configured Nugen model.

You will learn how to:
- Map task types to model identifiers.
- Call the Nugen chat-completions API.
- Keep API credentials outside the notebook.
- Handle HTTP errors and request timeouts.
- Test routing decisions without making API calls.

**Important:** This is a rule-based example, not automatic routing provided by Nugen. The model mapping is illustrative; confirm that the models you configure are available to your account and suitable for your workload.

Documentation: [Nugen API reference](https://docs.nugen.in/api-reference/) ? [Available language models](https://docs.nugen.in/model-reference/list_available_llms)


## 1. Install the dependency

The example uses `requests` to send HTTPS requests. Run the next cell in your notebook environment.

In [ ]:
%pip install -q requests

## 2. Configure your API key

Create an API key through the [Nugen Platform](https://platform.nugen.in/).

Set `NUGEN_API_KEY` in your environment before starting Jupyter. Do not paste a real key into a notebook cell or commit it to Git.

For example, in PowerShell, set it for the current terminal session before launching Jupyter:

```powershell
$env:NUGEN_API_KEY = "your-key-here"
```

Replace the placeholder locally; never publish the key.

In [ ]:
import os
import requests

API_KEY = os.getenv("NUGEN_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "NUGEN_API_KEY is not set. Set it in your environment and restart the notebook kernel."
    )

CHAT_COMPLETIONS_URL = (
    "https://api.nugen.in/api/v3/inference/chat/completions"
)


## 3. Define the routing policy

The model names below are examples listed in Nugen's model reference. Access and availability may vary. Change these values to models enabled for your account.

This example uses:
- `nugen-flash-instruct` for general questions.
- `qwen2p5-coder-32b-instruct` for coding tasks.
- `llama-v3p1-70b-instruct` for tasks explicitly marked as reasoning.

These labels are a routing policy chosen by the application, not a guarantee of model performance.

In [ ]:
MODEL_BY_TASK = {
    "general": "nugen-flash-instruct",
    "coding": "qwen2p5-coder-32b-instruct",
    "reasoning": "llama-v3p1-70b-instruct",
}

def classify_task(prompt: str) -> str:
    """Choose a task category using transparent keyword rules."""
    text = prompt.lower()

    coding_terms = (
        "write code", "debug", "python", "javascript",
        "stack trace", "function", "programming"
    )
    reasoning_terms = (
        "compare alternatives", "reason step by step",
        "analyze the trade-offs", "solve this logic problem"
    )

    if any(term in text for term in coding_terms):
        return "coding"
    if any(term in text for term in reasoning_terms):
        return "reasoning"
    return "general"

def select_model(prompt: str) -> tuple[str, str]:
    """Return the task category and its configured model identifier."""
    task = classify_task(prompt)
    return task, MODEL_BY_TASK[task]


## 4. Test routing without calling the API

These local tests check the routing policy only. They do not test model quality or require an API key.

In [ ]:
test_cases = [
    ("Explain what an API is.", "general"),
    ("Write Python code to sort a list.", "coding"),
    ("Compare alternatives and analyze the trade-offs.", "reasoning"),
]

for prompt, expected_task in test_cases:
    actual_task = classify_task(prompt)
    assert actual_task == expected_task, (
        f"Prompt {prompt!r}: expected {expected_task}, got {actual_task}"
    )
    print(f"{actual_task:10} -> {select_model(prompt)[1]}")

print("All routing tests passed.")


## 5. Make a request to Nugen

The function below sends a non-streaming chat-completions request. It uses a timeout, checks HTTP status codes, and reports a short error body if the request fails.

The function does not silently switch models after a failure: automatic retries or fallbacks should be designed carefully, with clear rules for rate limits, timeouts, cost, and model compatibility.

In [ ]:
def generate_response(prompt: str, max_tokens: int = 300) -> dict:
    """Route a prompt and return the selected model and API response."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string")
    if not isinstance(max_tokens, int) or not 1 <= max_tokens <= 4096:
        raise ValueError("max_tokens must be an integer between 1 and 4096")

    task, model = select_model(prompt)
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.2,
        "stream": False,
    }
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    try:
        response = requests.post(
            CHAT_COMPLETIONS_URL,
            json=payload,
            headers=headers,
            timeout=(10, 90),
        )
        response.raise_for_status()
    except requests.Timeout as exc:
        raise RuntimeError(
            "The Nugen request timed out. Check connectivity and try again."
        ) from exc
    except requests.HTTPError as exc:
        status = exc.response.status_code if exc.response is not None else "unknown"
        body = exc.response.text[:1000] if exc.response is not None else ""
        raise RuntimeError(
            f"Nugen returned HTTP {status}. Response excerpt: {body}"
        ) from exc
    except requests.RequestException as exc:
        raise RuntimeError(f"Could not reach the Nugen API: {exc}") from exc

    try:
        data = response.json()
    except ValueError as exc:
        raise RuntimeError("Nugen returned a response that was not valid JSON.") from exc

    return {"task": task, "model": model, "response": data}


## 6. Run an example

Run this cell only after setting your API key and verifying that the configured models are available to your account. This cell makes a real API request and may consume credits.

In [ ]:
result = generate_response(
    "Write Python code to check whether a string is a palindrome."
)

print("Selected task:", result["task"])
print("Selected model:", result["model"])
print("API response:")
print(result["response"])


## 7. Evaluate and improve the routing policy

A keyword router is a useful starting point, but it can misclassify ambiguous prompts. Before using a router in production:

1. Build a representative test set from your application's real tasks.
2. Measure task-classification accuracy and end-to-end response quality.
3. Compare latency, reliability, and cost for each candidate model.
4. Add explicit handling for rate limits and transient server errors.
5. Consider a safe fallback only when the fallback model is configured, available, and appropriate for the task.
6. Log request metadata and outcomes without recording secrets or unnecessary sensitive prompt content.

For production workloads, keep the model mapping in configuration and version the routing policy. Do not assume a larger model is always better.

## Conclusion

This recipe demonstrates a transparent, configurable application-level router using the Nugen chat-completions API. The routing rules are intentionally simple so they can be replaced with evaluation-driven rules as requirements grow.
